# MERFISH 500-Gene Panel Compression

This notebook starts from the ABC Atlas tutorial cache/API. It does not use a custom data loader. You read `adata` with `AbcProjectCache` and `anndata.read_h5ad`, then pass the prepared AnnData object into the training/evaluation helpers.

In [ ]:
from pathlib import Path
import sys

import anndata
import numpy as np
import pandas as pd
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

PROJECT_DIR = Path("..").resolve()
sys.path.insert(0, str(PROJECT_DIR))

from src.notebook import (
    evaluate_panels,
    learned_panels,
    merge_panel_dicts,
    run_baselines,
    train_panel_sizes,
)
from src.visualization.plots import plot_panel_comparison


In [ ]:
# Use the cache path from your working environment.
download_base = Path("/data/abc_atlas")
abc_cache = AbcProjectCache.from_cache_dir(download_base)
abc_cache.current_manifest


## Read MERFISH Exactly Like The ABC Tutorial

In [ ]:
file = abc_cache.get_file_path(
    directory="MERFISH-C57BL6J-638850",
    file_name="C57BL6J-638850/log2",
)
print(file)

adata = anndata.read_h5ad(file, backed="r")
adata


In [ ]:
cell = abc_cache.get_metadata_dataframe(
    directory="MERFISH-C57BL6J-638850",
    file_name="cell_metadata",
    dtype={"cell_label": str},
)
cell.set_index("cell_label", inplace=True)

cluster_details = abc_cache.get_metadata_dataframe(
    directory="WMB-taxonomy",
    file_name="cluster_to_cluster_annotation_membership_pivoted",
    keep_default_na=False,
)
cluster_details.set_index("cluster_alias", inplace=True)

cluster_colors = abc_cache.get_metadata_dataframe(
    directory="WMB-taxonomy",
    file_name="cluster_to_cluster_annotation_membership_color",
)
cluster_colors.set_index("cluster_alias", inplace=True)

cell_extended = cell.join(cluster_details, on="cluster_alias")
cell_extended = cell_extended.join(cluster_colors, on="cluster_alias")
cell_extended.head()


## Create The AnnData Object For This Experiment

In [ ]:
# Start small while developing. Set BRAIN_SECTION = None only when you are ready for all cells.
BRAIN_SECTION = "C57BL6J-638850.38"
MAX_CELLS = None
CELLTYPE_KEY = "subclass"

if BRAIN_SECTION is None:
    selected_cells = cell_extended.index
else:
    selected_cells = cell_extended.index[cell_extended["brain_section_label"] == BRAIN_SECTION]

if MAX_CELLS is not None and len(selected_cells) > MAX_CELLS:
    selected_cells = pd.Index(selected_cells).to_series().sample(MAX_CELLS, random_state=0).index

real_gene_mask = ~adata.var["gene_symbol"].astype(str).str.contains(
    "Blank|blank|codeword", regex=True, na=False
)

adata_exp = adata[selected_cells, adata.var.index[real_gene_mask]].to_memory()
adata_exp.obs = cell_extended.loc[adata_exp.obs_names].copy()
adata_exp.var["highly_variable"] = True
adata_exp.obsm["spatial"] = adata_exp.obs[["x", "y", "z"]].to_numpy(np.float32)

adata_exp = adata_exp[adata_exp.obs[CELLTYPE_KEY].notna()].copy()
celltypes = sorted(adata_exp.obs[CELLTYPE_KEY].unique())
ct_to_int = {ct: i for i, ct in enumerate(celltypes)}
adata_exp.obs["ct_label"] = adata_exp.obs[CELLTYPE_KEY].map(ct_to_int).astype(int)
adata_exp.uns["celltypes"] = celltypes
adata_exp.uns["ct_to_int"] = ct_to_int
adata_exp.uns["is_merfish"] = True

adata_exp


## Train And Evaluate Panels

In [ ]:
RESULTS_DIR = PROJECT_DIR / "results" / "merfish_compression"
PANEL_SIZES = [10, 25, 50, 100]

baseline_panels = run_baselines(
    adata_exp,
    panel_sizes=PANEL_SIZES,
    celltype_key=CELLTYPE_KEY,
    include_spapros=False,
)


In [ ]:
runs = train_panel_sizes(
    adata_exp,
    panel_sizes=PANEL_SIZES,
    output_dir=RESULTS_DIR / "trained_models",
    n_epochs=50,
    batch_size=512,
    num_workers=0,
    seed=0,
)

{k: v["selected_gene_symbols"] for k, v in runs.items()}


In [ ]:
all_panels = merge_panel_dicts(baseline_panels, learned_panels(runs))
results = evaluate_panels(
    adata_exp,
    all_panels,
    celltype_key="ct_label",
    run_clustering_min_k=25,
    output_csv=RESULTS_DIR / "eval_summary.csv",
)
results


In [ ]:
metrics = [m for m in ["f1_macro", "knn_overlap_mean", "nmi_mean"] if m in results.columns]
fig = plot_panel_comparison(
    results,
    metrics=metrics,
    save_path=str(RESULTS_DIR / "metric_comparison.png"),
)
fig
